# Offline Reinforcement Learning

In this notebook we'll **learn a policy from a fixed dataset of logged experience** (an _offline_
dataset) and evaluate it in a continuous-control MuJoCo task.

We'll start with **Behavioral Cloning (BC)** as a simple baseline (pure imitation learning), then
extend to a lightweight but faithful implementation of **Implicit Q-Learning (IQL)** - a practical
offline RL algorithm that learns from the same dataset while being more robust to the classic
offline RL failure mode: **out-of-distribution (OOD) actions**.


## What is Offline RL?

In standard _online_ RL, the agent alternates between:

1. Collecting new experience by interacting with the environment, and
2. Updating its policy from that experience.

In **Offline RL**, we remove step (1): we assume we have a **fixed dataset** of transitions
collected by some _behavior policy_ (e.g., a deployed controller, a human, or a previous RL agent),
and we must learn _only_ from that dataset:

- `s`: state
- `a`: action
- `r`: reward
- `s'`: next state
- `done`: episode termination flag

### What is Imitation Learning?

**Imitation Learning (IL)** is a paradigm where the agent learns a policy by **mimicking** a dataset
of demonstrations provided by an expert. Unlike RL, which learns from trial and error to maximize a
reward signal, IL assumes the demonstrator is (mostly) optimal and tries to copy their behavior.

**Behavioral Cloning (BC)** is the simplest form of IL. It treats the problem as **Supervised
Learning**: given a dataset of `(s, a)` pairs, it trains a policy $\pi(s)$ to predict action `a`
given state `s` (minimizing, for example, the Mean Squared Error).

- **Pros:** Simple, stable, and requires no reward function.
- **Cons:** It cannot improve over the demonstrator. If the dataset contains mixed-quality data
  (expert + random noise), BC will just copy the "average" behavior, which might be suboptimal.

### The Offline RL Challenge: Distribution Shift

Offline RL aims to do better than BC by using rewards to **select** the best actions in the dataset.
However, standard RL algorithms (like DQN or SAC) fail in offline settings due to **distribution
shift**.

Value-based updates often overestimate the value of **out-of-distribution (OOD) actions** (actions
not seen in the dataset). If the policy exploits these errors, it "hallucinates" high rewards for
crazy actions, and performance collapses.

**IQL (Implicit Q-Learning)** is designed to solve this by learning from the dataset while strictly
avoiding OOD actions.


In [ ]:
import random
from dataclasses import dataclass
from typing import Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

from util.gymnastics import DEVICE, gym_simulation, init_random
from util.rl_algos import AgentSAC

### Setup and Environment

We'll use **Swimmer-v5** (continuous control, MuJoCo).


In [ ]:
env = gym.make("Swimmer-v5")
env = init_random(env)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

print(f"Environment: Swimmer-v5")
print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")

### Loading the Expert and Visualizing its Performance

We'll load a pre-trained **SAC** expert to generate demonstration data and to use as an upper bound
for performance.


In [ ]:
expert_agent = AgentSAC(state_dim, action_dim)
expert_agent.load("solution/swimmer_v5_sac_weights.pth")
print("Expert agent loaded successfully.")

Let's see how our expert performs! The cell below will render the environment and show the expert
agent in action.


In [ ]:
# Visualize the expert's behavior
gym_simulation("Swimmer-v5", expert_agent, max_t=150)

### Creating an Offline Dataset

Offline RL assumes we cannot gather new data during training. So first we will build a **fixed
dataset** of transitions.

#### Let's make the dataset _suboptimal_...!

If we collect data from a perfect expert, then BC will often look great already, and offline RL
won't feel very necessary. Instead, we'll create a **behavior policy** that is _mostly_ expert, but
sometimes:

- Takes a random action, or
- Adds small Gaussian noise to the expert action.

This produces a more realistic offline dataset with mixed-quality behavior:

- BC will imitate the _average_ behavior in the dataset,
- IQL can learn to **prefer the better actions** found in the dataset (without proposing unseen
  actions).


In [ ]:
def generate_offline_dataset(
    env,
    expert_policy,
    num_episodes: int = 120,
    expert_action_prob: float = 0.7,
    action_noise_std: float = 0.05,
    seed: int = 0,
):
    """Collect a *fixed* offline dataset of transitions.

    Offline RL assumes you **cannot** collect new data while learning: you must learn
    from a static dataset D = {(s, a, r, s", done)}.

    We"ll build such a dataset by rolling out a **behavior policy** that is a mixture:
      - with probability `expert_action_prob`: take the expert action (optionally + noise)
      - otherwise: take a random action

    This creates a dataset that is *not perfect*, so it"s easier to see the difference
    between:
      - Behavioral Cloning (BC): imitates the behavior in the dataset
      - Offline RL (IQL): tries to maximize return while staying close to the dataset
    """
    # TODO: Implement the data collection logic.
    # 1. Initialize lists for states, actions, rewards, next_states, dones.
    # 2. Loop over episodes:
    #    a. Reset environment.
    #    b. Loop until done:
    #       i. Decide whether to take expert action or random action (based on expert_action_prob).
    #       ii. Step the environment.
    #       iii. Store the transition (s, a, r, s", done).
    # 3. Return the dataset as a dictionary of numpy arrays.
    return None


offline_data = generate_offline_dataset(
    env,
    expert_agent,
    num_episodes=120,
    expert_action_prob=0.7,
    action_noise_std=0.05,
    seed=0,
)

print("Offline dataset collected!")
print(f"Transitions: {len(offline_data["states"]):,}")
print(f"Episodes: {len(offline_data["episode_returns"]):,}")
print(
    f"Dataset mean episode return: {offline_data["episode_returns"].mean():.2f} ± "
    f"{offline_data["episode_returns"].std():.2f}"
)

### Dataset diagnostics (sanity checks)

Before training anything, it’s useful to **look at what’s actually inside the offline dataset**.
These plots help with both:

- **Teaching** (you can _see_ what “mixed-quality data” looks like), and
- **Debugging** (if BC/IQL behave oddly, the dataset is often the reason).

We’ll inspect:

- episode return distribution (how “good” the dataset is),
- episode lengths,
- action magnitudes / per-dimension action distribution,
- and the fraction of expert vs random actions actually taken by the behavior policy.


In [ ]:
ep_returns = offline_data["episode_returns"]
ep_lengths = offline_data["episode_lengths"]

# Transition-level stats
actions = offline_data["actions"]
is_expert = offline_data.get("is_expert", None)  # (N,) float in {0,1}

action_norm = np.linalg.norm(actions, axis=1)

print(f"Transitions: {len(actions):,}")
print(
    f"Episode return: mean={ep_returns.mean():.2f}, std={ep_returns.std():.2f}, min={ep_returns.min():.2f}, max={ep_returns.max():.2f}"
)
print(
    f"Episode length: mean={ep_lengths.mean():.1f}, std={ep_lengths.std():.1f}, min={ep_lengths.min()}, max={ep_lengths.max()}"
)
print(f"Action L2 norm: mean={action_norm.mean():.3f}, std={action_norm.std():.3f}")

if is_expert is not None:
    print(f"Expert-action fraction (per transition): {is_expert.mean()*100:.1f}%")

# Plots
sns.set_style("whitegrid")
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

# Episode returns
sns.histplot(ep_returns, bins=30, kde=True, ax=axs[0, 0])
axs[0, 0].set_title("Episode return distribution")
axs[0, 0].set_xlabel("Return")

# Episode lengths
sns.histplot(ep_lengths, bins=30, kde=True, ax=axs[0, 1])
axs[0, 1].set_title("Episode length distribution")
axs[0, 1].set_xlabel("Length")

# Action magnitudes
sns.histplot(action_norm, bins=30, kde=True, ax=axs[1, 0])
axs[1, 0].set_title("Action L2-norm distribution (transition-level)")
axs[1, 0].set_xlabel("||a||")

# Expert vs random actions
if is_expert is not None:
    frac = float(is_expert.mean())
    df_mix = pd.DataFrame({"behavior": ["expert", "random"], "fraction": [frac, 1.0 - frac]})
    sns.barplot(data=df_mix, x="behavior", y="fraction", ax=axs[1, 1])
    axs[1, 1].set_ylim(0, 1)
    axs[1, 1].set_title("Behavior-policy mixture (transition-level)")
else:
    axs[1, 1].axis("off")
    axs[1, 1].text(0.5, 0.5, "No expert-mask available", ha="center", va="center")

plt.tight_layout()
plt.show()

# Optional: per-dimension action histogram (helpful if action_dim > 1)
if actions.shape[1] > 1:
    df_a = pd.DataFrame(actions, columns=[f"a[{i}]" for i in range(actions.shape[1])])
    df_long = df_a.melt(var_name="dim", value_name="action")
    plt.figure(figsize=(10, 4))
    sns.histplot(
        data=df_long,
        x="action",
        hue="dim",
        bins=40,
        element="step",
        stat="density",
        common_norm=False,
    )
    plt.title("Per-dimension action distribution")
    plt.tight_layout()
    plt.show()

### Behavioral Cloning (BC) baseline

BC treats imitation learning as supervised learning: predict the dataset action `a` from the state
`s`.

Even though BC is not "RL" (it ignores rewards), it is an extremely strong baseline in offline
settings:

- it is stable,
- and it never proposes OOD actions (it learns to imitate the dataset).


In [ ]:
class BCAgent(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(BCAgent, self).__init__()
        # TODO: Define the network (e.g. a simple MLP with ReLU activations)
        self.network = None

    def forward(self, state: torch.Tensor):
        # TODO: Implement forward pass
        return None

    @torch.no_grad
    def act(self, state: np.ndarray) -> np.ndarray:
        # TODO: Implement act method (numpy state -> tensor -> network -> numpy action)
        return None

In [ ]:
bc_agent = BCAgent(state_dim, action_dim)

# Convert offline dataset to PyTorch tensors
states_tensor = torch.tensor(offline_data["states"], dtype=torch.float32)
actions_tensor = torch.tensor(offline_data["actions"], dtype=torch.float32)

dataset = TensorDataset(states_tensor, actions_tensor)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

optimizer = optim.Adam(bc_agent.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

num_epochs = 25
bc_loss_history = []  # mean training loss per epoch
print("Training BC agent...")
for epoch in range(num_epochs):
    epoch_losses = []
    for s_batch, a_batch in dataloader:
        # TODO: Implement the BC training loop
        # 1. Forward pass: predict actions
        pred_actions = None
        # 2. Compute loss (MSE between predicted and actual actions)
        loss = None
        # 3. Backward pass and optimization step
        pass

        # (Optional) Store loss for logging if you implemented it
        # epoch_losses.append(loss.item())

    # bc_loss_history.append(np.mean(epoch_losses))
    # if (epoch + 1) % 5 == 0 or epoch == 0:
    #    print(f"Epoch {epoch+1:02d}/{num_epochs} | Loss: {np.mean(epoch_losses):.6f}")
    pass

#### Why does the BC loss plateau so high?

MSE barely dips below ~0.265. That's the **irreducible noise floor** from how we built the dataset:
a random action is taken 30% of the time, and Gaussian noise is added to the expert action the other
70% — so at the same state the dataset contains multiple inconsistent actions, which a deterministic
BC network cannot simultaneously fit. This is exactly the kind of mixed-quality signal where IQL's
advantage-weighting should shine: instead of averaging over all dataset actions at a state, it
biases toward the ones that led to higher return.


### Offline RL with Implicit Q-Learning (IQL)

BC copies _what the dataset does_. Offline RL tries to copy **only the good parts** of the dataset,
while staying close enough to avoid out-of-distribution (OOD) actions.

IQL (“Implicit Q-Learning”) is a popular algorithm because it avoids taking a `max` over unseen
actions (which causes instability). It works in three steps:

#### 1. Critic Learning ($Q$)

We learn a Q-function $Q(s,a)$ using standard TD learning, but strictly on dataset transitions
$(s,a,r,s') \sim \mathcal{D}$. Since we don't query the Q-function for unseen actions in the target,
this step is stable.

#### 2. Value Learning ($V$) via Expectiles

We need a value function $V(s)$ to serve as a baseline. In standard RL, $V(s) = \max_a Q(s,a)$. In
offline RL, the `max` is dangerous.

Instead, IQL estimates $V(s)$ using **expectile regression**. We want $V(s)$ to represent the "best"
actions in the dataset at state $s$, not just the average. Expectile regression minimizes an
asymmetric loss:

$$
\mathcal{L}_\tau(v) = \mathbb{E}_{(s,a)\sim\mathcal{D}} \left[ |\tau - \mathbf{1}(Q(s,a) - v < 0)| (Q(s,a) - v)^2 \right]
$$

- If $\tau = 0.5$, this is just mean regression (average).
- If $\tau > 0.5$ (e.g., $\tau=0.7$), it **penalizes underestimates more than overestimates**. This
  effectively pushes $V(s)$ toward the **upper tail** of the $Q$-value distribution in the dataset.

**Intuition:** Think of the distribution of $Q(s, a)$ for various actions $a$ in the dataset at
state $s$ as a "cloud" of values. By setting $\tau > 0.5$, $V(s)$ converges not to the center of
this cloud (mean), but to its upper edge. This allows $V(s)$ to approximate the value of the _best_
actions supported by the data, without ever querying OOD actions.

See the appendix for an in-depth explanation of the formula if necessary.


In [ ]:
def expectile_loss(u: torch.Tensor, tau: float) -> torch.Tensor:
    # TODO: Implement expectile loss
    # Loss = |tau - I(u < 0)| * u^2
    # Hint: use torch.where to implement the indicator/weight logic
    #       If u > 0 (underestimate), weight is tau.
    #       If u <= 0 (overestimate), weight is 1 - tau.
    return None

In [ ]:
def test_expectile_loss():
    """Sanity checks on `expectile_loss`.

    The expectile loss is a weighted squared error `|tau - 1(u < 0)| * u^2`, with `u = Q - V`.
    We verify two properties:

    - At `tau = 0.5`, weights are symmetric (0.5) and the loss reduces to `0.5 * u^2`.
    - At `tau > 0.5`, positive residuals (Q > V, i.e. underestimates) are penalized *more*
      than negative ones -- which is what pushes V(s) toward the upper tail of Q at s.
    """
    u = torch.tensor([2.0, -2.0, 1.0, -1.0])

    # Symmetric case: loss is just 0.5 * u^2 everywhere.
    loss_mid = expectile_loss(u, tau=0.5)
    assert torch.allclose(
        loss_mid, 0.5 * u.pow(2)
    ), f"tau=0.5 should reduce to 0.5 * u^2, got {loss_mid}"

    # Asymmetric case: positive residuals weighted `tau`, negative weighted `1 - tau`.
    loss_hi = expectile_loss(u, tau=0.7)
    expected = torch.tensor([0.7 * 4.0, 0.3 * 4.0, 0.7 * 1.0, 0.3 * 1.0])
    assert torch.allclose(loss_hi, expected), f"tau=0.7 asymmetric weighting failed: got {loss_hi}"

    # Higher penalty on underestimates than overestimates of equal magnitude.
    same_mag_pos = torch.tensor([1.0, 1.0])
    same_mag_neg = torch.tensor([-1.0, -1.0])
    assert expectile_loss(same_mag_pos, 0.7).sum() > expectile_loss(same_mag_neg, 0.7).sum()

    print("✅ expectile_loss tests passed!")


test_expectile_loss()

#### 3. Policy Extraction via Advantage-Weighting

Now we have $Q(s,a)$ and $V(s)$. The **advantage** of an action is $A(s,a) = Q(s,a) - V(s)$.

- If $A > 0$, the action is better than the learned baseline (it's "unusually good" relative to the
  dataset).
- If $A < 0$, it is worse.

We train the policy $\pi(a|s)$ using **Advantage-Weighted Regression (AWR)**. We behave like BC
(maximizing likelihood of dataset actions), but we **weight** each sample by its exponentiated
advantage:

$$
\max_\pi \mathbb{E}_{(s,a)\sim\mathcal{D}} \left[ \exp(\beta \cdot A(s,a)) \log \pi(a|s) \right]
$$

where $\beta$ is a temperature hyperparameter.

- High $\beta$: We aggressively copy only the very best actions (closer to `max`).
- Low $\beta$: We revert to standard BC (copy everything equally).

---

### Networks


In [ ]:
def mlp(in_dim: int, out_dim: int, hidden: int = 256, depth: int = 2, act=nn.ReLU):
    # TODO: Implement a simple MLP builder
    return None


class QNetwork(nn.Module):
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        # TODO: Initialize MLP for Q(s, a). Input dimension should be state_dim + action_dim.
        self.net = None

    def forward(self, s: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        # TODO: Implement forward pass. Concatenate s and a.
        return None


class VNetwork(nn.Module):
    def __init__(self, state_dim: int):
        super().__init__()
        # TODO: Initialize MLP for V(s).
        self.net = None

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        # TODO: Implement forward pass.
        return None


class DeterministicPolicy(nn.Module):
    """Simplified policy: deterministic tanh network."""

    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        # TODO: Initialize network mapping state -> action.
        self.net = None

    def forward(self, s: torch.Tensor) -> torch.Tensor:
        # TODO: Implement forward pass.
        return None

    @torch.no_grad()
    def act(self, state: np.ndarray) -> np.ndarray:
        # TODO: Implement act method.
        return None

#### Simplifications and a toggle for a more canonical policy update (optional)

The original IQL paper typically uses a **stochastic** policy (e.g., tanh-squashed Gaussian) and
updates it by maximizing the weighted log-likelihood as shown above.

For teaching simplicity, many notebooks replace this with a **deterministic** policy trained by a
**weighted MSE** on actions. This notebook supports _both_:

- **Simplified (deterministic + weighted MSE)**: Easier to read; works well for demonstration.
- **Canonical (stochastic + weighted log-likelihood)**: Closer to common IQL implementations and
  standard papers.

We’ll expose this as a single toggle (`IQL_POLICY_MODE`) in the code below so you can switch modes
without rewriting the algorithm.


In [ ]:
IQL_POLICY_MODE = "simplified"  # try: "canonical"


class GaussianTanhPolicy(nn.Module):
    """More canonical policy: tanh-squashed Gaussian.

    We train it with weighted log-likelihood on dataset actions.
    """

    def __init__(
        self, state_dim: int, action_dim: int, log_std_min: float = -5.0, log_std_max: float = 2.0
    ):
        super().__init__()
        self.action_dim = action_dim
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max
        self.net = mlp(state_dim, 2 * action_dim)

    def forward(self, s: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        out = self.net(s)
        mean, log_std = out[..., : self.action_dim], out[..., self.action_dim :]
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mean, log_std

    def _atanh(self, x: torch.Tensor) -> torch.Tensor:
        # stable atanh for actions in (-1, 1)
        x = torch.clamp(x, -0.999, 0.999)
        return 0.5 * (torch.log1p(x) - torch.log1p(-x))

    def log_prob(self, s: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        """Log pi(a|s) for tanh-squashed Gaussian (with change-of-variables)."""
        mean, log_std = self.forward(s)
        std = torch.exp(log_std)
        u = self._atanh(a)

        # Normal log-prob in pre-tanh space
        normal = torch.distributions.Normal(mean, std)
        logp_u = normal.log_prob(u).sum(dim=-1)

        # Tanh change-of-variables term: sum log(1 - tanh(u)^2) = log(1 - a^2)
        eps = 1e-6
        log_det = torch.log(1.0 - a.pow(2) + eps).sum(dim=-1)
        return logp_u - log_det

    @torch.no_grad()
    def act(self, state: np.ndarray, deterministic: bool = True) -> np.ndarray:
        s = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(next(self.parameters()).device)
        mean, log_std = self.forward(s)
        if deterministic:
            u = mean
        else:
            std = torch.exp(log_std)
            u = mean + std * torch.randn_like(mean)
        a = torch.tanh(u)
        return a.squeeze(0).cpu().numpy()

### IQL (Implicit Q-Learning) Implementation


In [ ]:
@dataclass
class IQLConfig:
    gamma: float = 0.99
    tau: float = 0.7  # expectile level
    beta: float = 3.0  # advantage temperature for policy weights
    max_weight: float = 100.0  # clamp exp weights for stability
    lr: float = 3e-4
    batch_size: int = 256
    epochs: int = 60
    polyak: float = 0.005  # target update rate for V target


cfg = IQLConfig()

In [ ]:
# Convert data to tensors
s = torch.tensor(offline_data["states"], dtype=torch.float32).to(DEVICE)
a = torch.tensor(offline_data["actions"], dtype=torch.float32).to(DEVICE)
r = torch.tensor(offline_data["rewards"], dtype=torch.float32).to(DEVICE)
s2 = torch.tensor(offline_data["next_states"], dtype=torch.float32).to(DEVICE)
d = torch.tensor(offline_data["dones"], dtype=torch.float32).to(DEVICE)

N = s.shape[0]

# Networks
q1 = QNetwork(state_dim, action_dim).to(DEVICE)
q2 = QNetwork(state_dim, action_dim).to(DEVICE)
v = VNetwork(state_dim).to(DEVICE)
v_targ = VNetwork(state_dim).to(DEVICE)
v_targ.load_state_dict(v.state_dict())

if IQL_POLICY_MODE == "canonical":
    pi = GaussianTanhPolicy(state_dim, action_dim).to(DEVICE)
else:
    pi = DeterministicPolicy(state_dim, action_dim).to(DEVICE)

# Optimizers
opt_q = optim.Adam(list(q1.parameters()) + list(q2.parameters()), lr=cfg.lr)
opt_v = optim.Adam(v.parameters(), lr=cfg.lr)
opt_pi = optim.Adam(pi.parameters(), lr=cfg.lr)

# Loss histories for learning curves
iql_q_loss_history = []
iql_v_loss_history = []
iql_pi_loss_history = []

print(f"Training IQL agent... (policy mode: {IQL_POLICY_MODE})")
for epoch in tqdm(range(cfg.epochs), desc="IQL training"):
    q_losses, v_losses, pi_losses = [], [], []
    perm = torch.randperm(N, device=DEVICE)

    for start in range(0, N, cfg.batch_size):
        idx = perm[start : start + cfg.batch_size]
        s_b, a_b, r_b, s2_b, d_b = s[idx], a[idx], r[idx], s2[idx], d[idx]

        # 1. Critic update
        with torch.no_grad():
            # TODO: Compute target Q using V_target (Bellman equation: r + gamma * (1-d) * V_next)
            q_target = None

        # TODO: Compute predictions from both Q networks and the MSE loss
        q1_pred = None
        q2_pred = None
        q_loss = None

        # TODO: Optimize Q networks
        pass

        # 2. Value update (Expectile Regression)
        with torch.no_grad():
            # TODO: Compute Q_min = min(Q1, Q2)
            q_min = None

        # TODO: Compute V prediction and Expectile Loss (using expectile_loss function)
        v_pred = None
        v_loss = None

        # TODO: Optimize V network
        pass

        # 3. Policy update (Advantage Weighted)
        with torch.no_grad():
            # TODO: Compute advantage = Q_min - V
            adv = None
            # TODO: Compute weights = exp(beta * advantage) (clamp for stability)
            weights = None

        if IQL_POLICY_MODE == "canonical":
            # Weighted log-likelihood
            logp = pi.log_prob(s_b, a_b)
            pi_loss = -(weights * logp).mean()
        else:
            # Weighted MSE (simplified)
            # TODO: Compute BC MSE loss weighted by advantage weights
            a_pred = None
            pi_loss = None

        # TODO: Optimize Policy network
        pass

        # 4. Target update
        with torch.no_grad():
            # TODO: Soft update V_target towards V
            pass

        q_losses.append(q_loss.item())
        v_losses.append(v_loss.item())
        pi_losses.append(pi_loss.item())

    # Store epoch means
    iql_q_loss_history.append(float(np.mean(q_losses)))
    iql_v_loss_history.append(float(np.mean(v_losses)))
    iql_pi_loss_history.append(float(np.mean(pi_losses)))

    if (epoch + 1) % 10 == 0 or epoch == 0:
        pass
        # print(f"Epoch {epoch+1:03d}/{cfg.epochs} ...")

iql_agent = pi  # for evaluation

### Training curves (learning dynamics)

Final evaluation scores are useful, but it’s also instructive to see **how training behaved**. Below
we plot:

- BC training loss vs epoch
- IQL losses (Q, V, and policy loss) vs epoch

These curves can reveal instability (e.g., exploding weights) or slow learning (e.g., too-small
$\beta$).


In [ ]:
sns.set_style("whitegrid")

# BC curve
plt.figure(figsize=(10, 4))
sns.lineplot(x=np.arange(1, len(bc_loss_history) + 1), y=bc_loss_history)
plt.title("Behavioral Cloning training loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.tight_layout()
plt.show()

# IQL curves
df_iql = pd.DataFrame(
    {
        "epoch": np.arange(1, len(iql_q_loss_history) + 1),
        "Q loss": iql_q_loss_history,
        "V loss": iql_v_loss_history,
        "Policy loss": iql_pi_loss_history,
    }
)
df_long = df_iql.melt(id_vars="epoch", var_name="metric", value_name="value")

plt.figure(figsize=(10, 4))
sns.lineplot(data=df_long, x="epoch", y="value", hue="metric")
plt.title(f"IQL training losses (policy mode: {IQL_POLICY_MODE})")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

### Evaluating and Comparing Policies

Strictly speaking, in offline RL you often **cannot** evaluate online during training (e.g., a real
robot or production system). But in benchmark environments, we _can_ run evaluation episodes to
understand how well the learned policy works.

#### A Note on Real-World Offline Evaluation

In **true offline RL settings** (e.g., healthcare, industrial control), deploying an untested policy
can be dangerous or expensive. Therefore, practitioners rely on **Off-Policy Evaluation (OPE)**
techniques to estimate performance without environment interaction, such as:

- **Fitted Q Evaluation (FQE):** Training a specific critic to estimate the value of the new policy
  $\pi$ using the static dataset.
- **Importance Sampling (IS):** Weighting historical returns by the probability ratio
  $\frac{\pi(a|s)}{\pi_{\text{behavior}}(a|s)}$ (though this often suffers from high variance).
- **Holdout Validation:** Monitoring prediction errors (like TD-error or BC loss) on a held-out
  validation set of transitions.

Here, we'll use the simulator to get the ground-truth performance comparing:

- the **Expert** (upper bound),
- **BC** (offline imitation baseline),
- **IQL** (offline RL from the same dataset).


In [ ]:
def evaluate_policy(env, policy, num_episodes=50):
    total_rewards = []
    for _ in tqdm(range(num_episodes), desc=f"Evaluating {policy.__class__.__name__}"):
        state, _ = env.reset()
        done = False
        episode_reward = 0

        while not done:
            action = policy.act(state)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward

        total_rewards.append(episode_reward)

    return total_rewards

In [ ]:
def summarize(rs):
    return float(np.mean(rs)), float(np.std(rs))


# Evaluate policies
expert_rewards = evaluate_policy(env, expert_agent, num_episodes=50)
bc_rewards = evaluate_policy(env, bc_agent, num_episodes=50)
iql_rewards = evaluate_policy(env, iql_agent, num_episodes=50)

expert_mean, expert_std = summarize(expert_rewards)
bc_mean, bc_std = summarize(bc_rewards)
iql_mean, iql_std = summarize(iql_rewards)


print(f"Expert Performance: {expert_mean:.2f} ± {expert_std:.2f}")
print(f"BC Performance:     {bc_mean:.2f} ± {bc_std:.2f}")
print(f"IQL Performance:    {iql_mean:.2f} ± {iql_std:.2f}")

### Performance Visualization

We plot the full distribution of episode returns to compare the stability and peak performance of
the learned policies.


In [ ]:
sns.set_style("whitegrid")
plt.figure(figsize=(10, 5))

sns.violinplot(
    data=[expert_rewards, bc_rewards, iql_rewards],
    palette=["#1f77b4", "#ff7f0e", "#12ad0e"],
)

plt.xticks([0, 1, 2], ["Expert", "BC Agent", "IQL Agent"])
plt.title("Distribution of Episode Rewards: Offline Learning comparison")
plt.ylabel("Episode Reward")
plt.tight_layout()
plt.show()

### Visualizing the Learned Policies

Let's watch the BC and IQL agents in action.


In [ ]:
# Visualize BC
gym_simulation("Swimmer-v5", bc_agent, max_t=150)

In [ ]:
# Visualize IQL
gym_simulation("Swimmer-v5", iql_agent, max_t=150)

## Further exploration

This notebook intentionally focuses on a minimal, educational path:

- **BC** as a strong imitation baseline
- **IQL** as a robust offline RL method that avoids hard OOD action maximization

If you want to go deeper, here are a few directions that build naturally from what you already have
here.

### Things to try _within this notebook_

- **Vary dataset quality:** change `expert_action_prob` and/or `action_noise_std` in
  `generate_offline_dataset`.  
  As the dataset gets cleaner, BC should close the gap; as it gets noisier or more mixed,
  advantage-weighted methods tend to help more.
- **Switch the IQL policy update:** flip `IQL_POLICY_MODE` between `"simplified"` and `"canonical"`
  and compare: deterministic weighted-MSE vs stochastic weighted log-likelihood.
- **Tune the two most influential IQL knobs:**
  - `tau` (expectile): higher values push `V(s)` upward toward “better” dataset actions.
  - `beta` (temperature): higher values put more weight on high-advantage actions (but can saturate
    / get unstable).
- **Inspect the failure mode directly:** if you make the dataset very suboptimal, watch how an
  unconstrained policy update can try to exploit Q-function errors; the advantage-weighting +
  dataset constraint is the key safeguard offline.

### More offline RL algorithms

- **TD3+BC:** a straightforward actor-critic baseline where the policy is regularized toward dataset
  actions. Often a strong first comparison point.
- **CQL (Conservative Q-Learning):** explicitly _pushes down_ Q-values for unseen actions to reduce
  OOD overestimation.
- **AWAC / AWR:** advantage-weighted behavior cloning variants closely related to the policy
  extraction step used here.
- **BCQ / BEAR:** older but historically important approaches that constrain actions to remain
  in-support of the dataset.
- **Flow-based / diffusion policies** can model multi-modal action distributions and are
  increasingly used in imitation + offline RL.


## Appendix

### Expectile regression formula

The formula is:

$$
\mathcal{L}_\tau(v) = \mathbb{E} \left[ |\tau - \mathbf{1}(Q - v < 0)| (Q - v)^2 \right]
$$

Let's define the residual (the error) as $u = Q - v$. The formula is essentially calculating a
Weighted Mean Squared Error:

$$
\text{Loss} = \text{Weight} \times u^2
$$

The term $\mathbf{1}(u < 0)$ is an indicator function. It is 1 if the condition inside is true, and
0 if false. Let's assume $\tau = 0.7$ (a common choice for IQL).

Case A: The target $Q$ is larger than your estimate $v$ ($Q > v$)

- This is an underestimate ($u > 0$).
- The condition $(Q - v < 0)$ is False.
- The indicator $\mathbf{1}(\dots)$ becomes 0.
- The weight becomes: $|\tau - 0| = \tau = \mathbf{0.7}$.

Case B: The target $Q$ is smaller than your estimate $v$ ($Q < v$)

- This is an overestimate ($u < 0$).
- The condition $(Q - v < 0)$ is True.
- The indicator $\mathbf{1}(\dots)$ becomes 1.
- The weight becomes: $|\tau - 1| = |0.7 - 1| = |-0.3| = \mathbf{0.3}$. (which is $1-\tau$).

Because $\tau > 0.5$, we pay a higher penalty (0.7) for being too low (Case A) than we pay (0.3) for
being too high (Case B). To minimize this loss, the model learns to push its predictions upwards to
avoid that expensive underestimation penalty. This is why the learned value $V(s)$ settles near the
top of the distribution rather than the average.

The complex-looking $|\tau - \mathbf{1}|$ notation is just a fancy mathematical way of writing "use
weight $\tau$ if positive, and $1-\tau$ if negative".
